In [2]:
!pip install scikit-learn
!pip install scikit-multilearn
!pip install openpyxl
!pip install tensorflow
import numpy as np
import pandas as pd
from skmultilearn.model_selection import IterativeStratification
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import Sequence
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt


In [3]:


# loading the dataframe df_audio
df_audio = pd.read_excel("/work/nlpminifolder/extracted_files/df_audio.xlsx")
df_audio.head()

,filepath,filename,track_id,audio_path,top_genres,num_labels,label_Rock,label_Electronic,label_Pop,label_Folk,...,label_HipHop,label_International,label_Classical,label_Jazz,label_Country,label_Blues,label_SoulRnB,waveform_path,mel_path,cqt_path
0,fma_small/018/018032.mp3,018032.mp3,18032,/datasets/fma/fma_clean/fma_selected/018/01803...,[2],1,0,0,0,0,...,0,1,0,0,0,0,0,/work/nlpminifolder/extracted_files/waveforms/...,/work/nlpminifolder/extracted_files/mels/018/0...,/work/nlpminifolder/extracted_files/cqt/018032...
1,fma_small/018/018037.mp3,018037.mp3,18037,/datasets/fma/fma_clean/fma_selected/018/01803...,[2],1,0,0,0,0,...,0,1,0,0,0,0,0,/work/nlpminifolder/extracted_files/waveforms/...,/work/nlpminifolder/extracted_files/mels/018/0...,/work/nlpminifolder/extracted_files/cqt/018037...
2,fma_small/018/018144.mp3,018144.mp3,18144,/datasets/fma/fma_clean/fma_selected/018/01814...,[10],1,0,0,1,0,...,0,0,0,0,0,0,0,/work/nlpminifolder/extracted_files/waveforms/...,/work/nlpminifolder/extracted_files/mels/018/0...,/work/nlpminifolder/extracted_files/cqt/018144...
3,fma_small/018/018159.mp3,018159.mp3,18159,/datasets/fma/fma_clean/fma_selected/018/01815...,[17],1,0,0,0,1,...,0,0,0,0,0,0,0,/work/nlpminifolder/extracted_files/waveforms/...,/work/nlpminifolder/extracted_files/mels/018/0...,/work/nlpminifolder/extracted_files/cqt/018159...
4,fma_small/018/018038.mp3,018038.mp3,18038,/datasets/fma/fma_clean/fma_selected/018/01803...,[2],1,0,0,0,0,...,0,1,0,0,0,0,0,/work/nlpminifolder/extracted_files/waveforms/...,/work/nlpminifolder/extracted_files/mels/018/0...,/work/nlpminifolder/extracted_files/cqt/018038...


In [4]:
label_cols = [c for c in df_audio.columns if c.startswith("label_")]

genre_distribution = df_audio[label_cols].sum().sort_values(ascending=False)
print(genre_distribution)
# 5 genres missing, due to fma_small

label_International    1500
label_HipHop           1245
label_Folk             1236
label_Electronic       1191
label_Pop              1182
label_Rock             1113
label_Instrumental      612
label_Classical           0
label_Jazz                0
label_Country             0
label_Blues               0
label_SoulRnB             0
dtype: int64


In [ ]:
# Drop genres with zero samples (fma_small only covers 7 of 12 genres)
genre_counts = df_audio[label_cols].sum()
zero_genres = genre_counts[genre_counts == 0].index.tolist()
print("Genres with zero samples:", zero_genres)

df_audio = df_audio.drop(columns=zero_genres, errors="ignore")
print("\nRemaining labels:", [c for c in df_audio.columns if c.startswith("label_")])

In [ ]:
# 64% train, 16% validation, 20% test — multilabel-stratified
from skmultilearn.model_selection import iterative_train_test_split

SEED = 42
rng = np.random.default_rng(SEED)

label_cols = [c for c in df_audio.columns if c.startswith("label_")]
Y = df_audio[label_cols].values
X = np.arange(len(Y)).reshape(-1, 1)

X_train_temp, Y_train_temp, X_test, Y_test = iterative_train_test_split(X, Y, test_size=0.20)
train_temp_idx = X_train_temp.flatten()
test_idx = X_test.flatten()
rng.shuffle(train_temp_idx)
rng.shuffle(test_idx)

X_train, Y_train, X_val, Y_val = iterative_train_test_split(X_train_temp, Y_train_temp, test_size=0.20)
train_idx = X_train.flatten()
val_idx = X_val.flatten()
rng.shuffle(train_idx)
rng.shuffle(val_idx)

df_train = df_audio.iloc[train_idx].reset_index(drop=True)
df_val = df_audio.iloc[val_idx].reset_index(drop=True)
df_test = df_audio.iloc[test_idx].reset_index(drop=True)

print(f"Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")
print("\nLabel distribution check:")
print("Train:\n", df_train[label_cols].sum())
print("\nVal:\n", df_val[label_cols].sum())
print("\nTest:\n", df_test[label_cols].sum())

In [ ]:
# 1D CNN (waveform) and 2D CNN (mel / CQT / multichannel) share the same
# architecture depth. The 2D variant handles all spectrogram types — only
# the input shape (channels) differs.

In [ ]:
# 1D CNN for waveform inputs
def make_1d_cnn(num_labels, input_length=1323000):
    model = models.Sequential([
        layers.Conv1D(32, kernel_size=7, activation="relu", padding="same",
                      input_shape=(input_length, 1)),
        layers.MaxPooling1D(pool_size=4),
        layers.Conv1D(64, kernel_size=5, activation="relu", padding="same"),
        layers.MaxPooling1D(pool_size=4),
        layers.Conv1D(128, kernel_size=5, activation="relu", padding="same"),
        layers.MaxPooling1D(pool_size=4),
        layers.Conv1D(256, kernel_size=3, activation="relu", padding="same"),
        layers.MaxPooling1D(pool_size=4),
        layers.Flatten(),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(num_labels, activation="sigmoid"),
    ])
    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=[tf.keras.metrics.AUC(curve="ROC", name="auc")],
    )
    return model

In [ ]:
# 2D CNN for mel, CQT, and multichannel spectrogram inputs
import tensorflow as tf
from tensorflow.keras import layers, models

def make_2d_cnn(num_labels, input_shape=(128, 3000, 1)):
    model = models.Sequential([
        layers.Conv2D(32, kernel_size=(3, 3), activation="relu",
                      padding="same", input_shape=input_shape),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Conv2D(64, kernel_size=(3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Conv2D(128, kernel_size=(3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Conv2D(256, kernel_size=(3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Flatten(),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(num_labels, activation="sigmoid"),
    ])
    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=[tf.keras.metrics.AUC(curve="ROC", name="auc")],
    )
    return model

In [11]:
#training the models


In [ ]:
class GenericAudioDataset(Sequence):
    def __init__(self, df, x_col, y_cols, batch_size=16, shuffle=True):
        self.df = df
        self.x_paths = df[x_col].values
        self.y = df[y_cols].values.astype("float32")
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indices = np.arange(len(df))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, idx):
        batch_ids = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        X_batch = np.array([np.load(self.x_paths[i]) for i in batch_ids]).astype("float32")
        if X_batch.ndim == 2:
            X_batch = X_batch[..., np.newaxis]  # 1D waveform → (batch, n, 1)
        return X_batch, self.y[batch_ids]

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)


def earlystopping():
    return EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True, verbose=1)


def train_model(model, train_loader, val_loader, epochs=10):
    return model.fit(train_loader, validation_data=val_loader,
                     epochs=epochs, callbacks=[earlystopping()], verbose=1)


def evaluate_model(model, test_loader):
    results = model.evaluate(test_loader, verbose=1)
    print("\nTest results:", results)
    return results

In [ ]:
# Waveform / 1D CNN

In [ ]:
train_wave = GenericAudioDataset(df_train, "waveform_path", label_cols)
val_wave   = GenericAudioDataset(df_val,   "waveform_path", label_cols)
test_wave  = GenericAudioDataset(df_test,  "waveform_path", label_cols)

In [15]:
'''
#training the waveform/1D CNN
model_wave = make_1d_cnn(num_labels=len(label_cols))

history_wave = train_model(
    model_wave,
    train_wave,
    val_wave,
    epochs=10
)
'''

'\n#training the waveform/1D CNN\nmodel_wave = make_1d_cnn(num_labels=len(label_cols))\n\nhistory_wave = train_model(\n    model_wave,\n    train_wave,\n    val_wave,\n    epochs=10\n)\n'

In [ ]:
# Mel and CQT spectrograms

In [ ]:
train_mel = GenericAudioDataset(df_train, "mel_path", label_cols)
val_mel   = GenericAudioDataset(df_val,   "mel_path", label_cols)
test_mel  = GenericAudioDataset(df_test,  "mel_path", label_cols)

train_cqt = GenericAudioDataset(df_train, "cqt_path", label_cols)
val_cqt   = GenericAudioDataset(df_val,   "cqt_path", label_cols)
test_cqt  = GenericAudioDataset(df_test,  "cqt_path", label_cols)

In [19]:
#training mel spectogram
'''
model_mel = make_2d_cnn(len(label_cols), input_shape=(128,3000,1))

history_mel = train_model(
    model_mel,
    train_mel,
    val_mel,
    epochs=10
)
'''

'\nmodel_mel = make_2d_cnn(len(label_cols), input_shape=(128,3000,1))\n\nhistory_mel = train_model(\n    model_mel,\n    train_mel,\n    val_mel,\n    epochs=10\n)\n'

In [20]:
#training cqt spectogram
'''
model_cqt = make_2d_cnn(len(label_cols), input_shape=(128,3000,1))

history_cqt = train_model(
    model_cqt,
    train_cqt,
    val_cqt,
    epochs=10
)
'''

'\nmodel_cqt = make_2d_cnn(len(label_cols), input_shape=(128,3000,1))\n\nhistory_cqt = train_model(\n    model_cqt,\n    train_cqt,\n    val_cqt,\n    epochs=10\n)\n'

In [21]:
#multichannel specific

In [ ]:
# Multichannel: stack mel + CQT along the channel axis → (128, 3000, 2)
class MultiChannelAudioDataset(Sequence):
    def __init__(self, df, mel_col, cqt_col, y_cols, batch_size=16, shuffle=True):
        self.df = df
        self.mel_paths = df[mel_col].values
        self.cqt_paths = df[cqt_col].values
        self.y = df[y_cols].values.astype("float32")
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indices = np.arange(len(df))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, idx):
        batch_ids = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        X_batch = []
        for i in batch_ids:
            mel = np.load(self.mel_paths[i])
            cqt = np.load(self.cqt_paths[i])
            X_batch.append(np.stack([mel, cqt], axis=-1))
        return np.array(X_batch), self.y[batch_ids]

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

In [ ]:
train_multi = MultiChannelAudioDataset(df_train, mel_col="mel_path", cqt_col="cqt_path", y_cols=label_cols)
val_multi   = MultiChannelAudioDataset(df_val,   mel_col="mel_path", cqt_col="cqt_path", y_cols=label_cols)
test_multi  = MultiChannelAudioDataset(df_test,  mel_col="mel_path", cqt_col="cqt_path", y_cols=label_cols)

In [25]:
#training multichannel

model_multi = make_2d_cnn(
    num_labels=len(label_cols),
    input_shape=(128, 3000, 2)
)
history_multi = train_model(
    model_multi,
    train_multi,
    val_multi,
    epochs=10
)


/opt/conda/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10


/opt/conda/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


324/324 ━━━━━━━━━━━━━━━━━━━━ 131s 396ms/step - auc: 0.5579 - loss: 1.0167 - val_auc: 0.7188 - val_loss: 0.3733
Epoch 2/10
324/324 ━━━━━━━━━━━━━━━━━━━━ 125s 385ms/step - auc: 0.7148 - loss: 0.3763 - val_auc: 0.7377 - val_loss: 0.3641
Epoch 3/10
324/324 ━━━━━━━━━━━━━━━━━━━━ 125s 384ms/step - auc: 0.7533 - loss: 0.3589 - val_auc: 0.7648 - val_loss: 0.3527
Epoch 4/10
324/324 ━━━━━━━━━━━━━━━━━━━━ 125s 386ms/step - auc: 0.8023 - loss: 0.3332 - val_auc: 0.7766 - val_loss: 0.3453
Epoch 5/10
324/324 ━━━━━━━━━━━━━━━━━━━━ 125s 384ms/step - auc: 0.8633 - loss: 0.2863 - val_auc: 0.7961 - val_loss: 0.3600
Epoch 6/10
324/324 ━━━━━━━━━━━━━━━━━━━━ 105s 324ms/step - auc: 0.9247 - loss: 0.2193 - val_auc: 0.8031 - val_loss: 0.3545
Epoch 6: early stopping
Restoring model weights from the end of the best epoch: 4.


In [27]:
#saving training results

In [28]:
import os

SAVE_DIR = "/work/nlpminifolder/model_results"
os.makedirs(SAVE_DIR, exist_ok=True)


In [ ]:
import numpy as np
from tensorflow.keras.models import load_model

def save_model(model, history, name):
    model_path = f"{SAVE_DIR}/{name}.h5"
    history_path = f"{SAVE_DIR}/{name}_history.npy"
    model.save(model_path)
    np.save(history_path, history.history)
    print(f"Saved: {model_path}")


def load_saved_model(name):
    model_path = f"{SAVE_DIR}/{name}.h5"
    history_path = f"{SAVE_DIR}/{name}_history.npy"
    model = load_model(model_path)
    history = np.load(history_path, allow_pickle=True).item()
    print(f"Loaded: {model_path}")
    return model, history

In [ ]:
# Training runs are commented out — models loaded from saved .h5 files below
#save_model(model_wave, history_wave, "waveform")
#save_model(model_mel, history_mel, "mel")
#save_model(model_cqt, history_cqt, "cqt")
#save_model(model_multi, history_multi, "multichannel")

In [ ]:
# Load saved models and evaluate on test set

In [33]:
#loading saved training results
from tensorflow.keras.models import load_model
import numpy as np
# load trained data
model_wave = load_model("/work/nlpminifolder/model_results/waveform.h5")
model_mel = load_model("/work/nlpminifolder/model_results/mel.h5")
model_cqt = load_model("/work/nlpminifolder/model_results/cqt.h5")
model_multi = load_model("/work/nlpminifolder/model_results/multichannel.h5")

#further data extraction for plotting data etc later on
history_wave = np.load("/work/nlpminifolder/model_results/waveform_history.npy", allow_pickle=True).item()
history_mel = np.load("/work/nlpminifolder/model_results/mel_history.npy",allow_pickle=True).item()
history_multi = np.load("/work/nlpminifolder/model_results/multichannel_history.npy",allow_pickle=True).item()
history_cqt = np.load("/work/nlpminifolder/model_results/cqt_history.npy",allow_pickle=True).item()



In [34]:
#testing
#wave predictions
y_pred_1DCNN = model_wave.predict(test_wave)
# mel predictions
y_pred_mel = model_mel.predict(test_mel)
#cqt predictions
y_pred_cqt = model_cqt.predict(test_cqt)
# multichannel predictions
y_pred_multi = model_multi.predict(test_multi)

101/101 ━━━━━━━━━━━━━━━━━━━━ 21s 206ms/step
101/101 ━━━━━━━━━━━━━━━━━━━━ 7s 66ms/step
101/101 ━━━━━━━━━━━━━━━━━━━━ 7s 66ms/step
101/101 ━━━━━━━━━━━━━━━━━━━━ 8s 80ms/step


In [ ]:
threshold = 0.5

y_true_1DCNN = np.vstack([batch[1] for batch in test_wave])
y_pred_bin_1DCNN = (y_pred_1DCNN >= threshold).astype(int)

y_true_mel = np.vstack([batch[1] for batch in test_mel])
y_pred_bin_mel = (y_pred_mel >= threshold).astype(int)

y_true_cqt = np.vstack([batch[1] for batch in test_cqt])
y_pred_bin_cqt = (y_pred_cqt >= threshold).astype(int)

y_true_multi = np.vstack([batch[1] for batch in test_multi])
y_pred_bin_multi = (y_pred_multi >= threshold).astype(int)

In [37]:
# ---- Waveform ----
f1_micro_wave = f1_score(y_true_1DCNN, y_pred_bin_1DCNN, average='micro')
f1_macro_wave = f1_score(y_true_1DCNN, y_pred_bin_1DCNN, average='macro')
print("F1 Micro (Waveform 1D-CNN):", f1_micro_wave)
print("F1 Macro (Waveform 1D-CNN):", f1_macro_wave)

# ---- Mel ----
f1_micro_mel = f1_score(y_true_mel, y_pred_bin_mel, average='micro')
f1_macro_mel = f1_score(y_true_mel, y_pred_bin_mel, average='macro')
print("F1 Micro (Mel):", f1_micro_mel)
print("F1 Macro (Mel):", f1_macro_mel)

# ---- CQT ----
f1_micro_cqt = f1_score(y_true_cqt, y_pred_bin_cqt, average='micro')
f1_macro_cqt = f1_score(y_true_cqt, y_pred_bin_cqt, average='macro')
print("F1 Micro (CQT):", f1_micro_cqt)
print("F1 Macro (CQT):", f1_macro_cqt)

# ---- Multichannel ----
f1_micro_multi = f1_score(y_true_multi, y_pred_bin_multi, average='micro')
f1_macro_multi = f1_score(y_true_multi, y_pred_bin_multi, average='macro')
print("F1 Micro (Multichannel):", f1_micro_multi)
print("F1 Macro (Multichannel):", f1_macro_multi)


F1 Micro (Waveform 1D-CNN): 0.05486284289276808
F1 Macro (Waveform 1D-CNN): 0.03992744377535575
F1 Micro (Mel): 0.11860276198212835
F1 Macro (Mel): 0.09749983492330157
F1 Micro (CQT): 0.08832807570977919
F1 Macro (CQT): 0.07209808385826463
F1 Micro (Multichannel): 0.10496338486574451
F1 Macro (Multichannel): 0.09136169415965686


In [ ]:
# Result: mel spectrogram outperforms CQT, multichannel, and raw waveform